# Customer Segmentation

Customer segmentation is the process of dividing a business's customer base into distinct groups based on shared characteristics. These groups (or "segments") are created so that businesses can better target their marketing, sales, product development, and customer service efforts to meet specific needs.

### Why Customer Segmentation Matters
- Improves marketing effectiveness by tailoring messages to each group.
- Boosts customer retention through personalized experiences.
- Increases sales and ROI by targeting high-value segments.
- Helps product development by identifying unmet needs in specific groups.

## Common Types of Customer Segmentation

- Demographic: Based on age, gender, income, education, etc.	
- Geographic:	Based on location: country, region, city, climate	Urban vs rural customers
- Behavioral: Based on purchasing behavior, usage, loyalty, or product interactions	

## RFM Analysis for Customer Segmentation

**RFM** (Recency, Frequency, Monetary) analysis is a powerful technique used in customer segmentation, especially for understanding and predicting customer behavior based on past interactions.


| Component     | Description                             | Example                              |
| ------------- | --------------------------------------- | ------------------------------------ |
| **Recency (R)**   | How recently a customer made a purchase | Bought something 2 weeks ago         |
| **Frequency (F)** | How often a customer makes a purchase   | Bought 10 times in the last 6 months |
| **Monetary (M)**  | How much money a customer spends        | Spent \$500 in total                 |

### How RFM Segmentation Works

1. Collect data – Pull customer purchase history (dates, frequency, amounts).
2. Score each customer – Usually on a scale from 1 to 5 for each metric:
   - 5 = best (e.g., bought very recently, buys often, spends a lot)
   - 1 = least valuable (e.g., hasn’t bought in months, buys rarely, spends little)

3. Create RFM segments – Combine the scores to group customers:
   - 555: Best customers (most recent, most frequent, highest spenders)
   - 155: Big spenders who haven’t bought recently
   - 511: Recent first-time buyers
   - 111: At-risk or churned customers

4. Take action based on segment:
   - Reward loyal customers
   - Re-engage inactive customers
   - Upsell to frequent buyers



In [ ]:
!pip install openpyxl

### Implementation

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

#### Stept 1: Load and Explore the Data

In [ ]:
df = pd.read_excel('data/OnlineRetail.xlsx',sheet_name='OnlineRetail')

In [ ]:
df.shape

In [ ]:
df.head()

### Data Cleaning

In [ ]:
df.isna().sum()

Remove the Rows that have `CustomerID` missing, since we can not segment them     

In [ ]:
df = df.dropna(axis=0,subset=['CustomerID'])

In [ ]:
df.shape

In [ ]:
df.info()

Convert the InvoiceDate tot Datetime

In [ ]:
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])

`CustomerID` is assigned `float` data type which is wrong. We will make it string   

In [ ]:
df['CustomerID'] = df['CustomerID'].astype(str).str.replace(r'\.0$', '', regex=True)

Create a `TotalAmount` to compute Monetary value 

In [ ]:
# Create TotalAmount column
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

We have some Column that has negative Quantity which usually means returns we can filter them out

In [ ]:
df[df['TotalAmount']<0]

In [ ]:
df = df[df['TotalAmount']>0]

In [ ]:
df.shape

### Set the Snapshot Date

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

In [ ]:
snapshot_date

In [ ]:
df['InvoiceDate'].max()

### Calculate RFM Metrics

In [ ]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalAmount': 'sum'
}).reset_index()

In [ ]:
rfm.head()

In [ ]:
# Rename columns
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

In [ ]:
rfm.head()

In [ ]:
fig,ax = plt.subplots(nrows=3,ncols=1,figsize=(10,8))
sns.histplot(data=rfm,x='Recency',kde=True,ax=ax[0])
ax[0].set_title('Distribution of Recency')
ax[0].set_xlabel('Recency')
ax[0].set_ylabel('Count')
sns.histplot(data=rfm,x='Frequency',kde=True,ax=ax[1])
ax[1].set_title('Distribution of Frequency')
ax[1].set_xlabel('Frequency')
ax[1].set_ylabel('Count')
sns.histplot(data=rfm,x='Monetary',kde=True,ax=ax[2])
ax[2].set_title('Distribution of Monetary')
ax[2].set_xlabel('Monetary')
ax[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(rfm)

### Assign RFM Scores

In [ ]:
# 1. Score Recency (lower is better → higher score)
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1]).astype(int)


In [ ]:
# 2. Score Frequency (higher is better → higher score)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)

In [ ]:
# 3. Score Monetary (higher is better → higher score)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)

In [ ]:
rfm['RFM_Total'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

In [ ]:
rfm.head()

In [ ]:
def rfm_segment(row):
    if row['RFM_Total'] >= 13:
        return 'Champion'
    elif row['R_Score'] >= 4:
        return 'Loyal'
    elif row['RFM_Total'] <= 5:
        return 'Lost'
    elif row['F_Score'] >= 4:
        return 'Frequent Buyer'
    else:
        return 'Others'

In [ ]:
rfm['Segment'] = rfm.apply(rfm_segment, axis=1)

In [ ]:
rfm.head()

In [ ]:
rfm_count = rfm['Segment'].value_counts()

In [ ]:
rfm_count.plot(kind='bar')

In [ ]:
rfm.to_csv('data/rfm_analysis.csv',index=False)

In [ ]:
rfm.columns